# Experimentos reproducibles — Antes de que falle (SSD MC1)

**Semana 10 · Registro de experimentos con MLflow**

Este cuaderno compara cuatro modelos de **clasificación binaria** para estimar `TARGET_30D`, la presencia de una falla SSD durante los próximos 30 días. Utiliza la OBT `OBT_MC1_RN` del proyecto mediante la misma conexión Spark–Snowflake del cuaderno anterior.

El entregable se genera a partir de una muestra de **hasta 50.000 observaciones**. No se recalculan las ventanas SMART, no se imputan valores faltantes y ninguna columna de fecha, identidad o resultado futuro entra como predictor. El proceso conserva un archivo local con la muestra y registra su huella SHA-256 para reproducir exactamente los datos consultados.

**Modelos:** línea base (`DummyClassifier`), bosque aleatorio (`RandomForestClassifier`), boosting (`HistGradientBoostingClassifier`) y red neuronal (`MLPClassifier`). El bosque aleatorio sustituye al modelo lineal por decisión de este proyecto. Esta sustitución debe indicarse si la consigna exige literalmente un modelo lineal.

## 0. Preparar el entorno

Coloca `ExperimentosModelos_MC1_Proyecto.ipynb`, `pipeline_proyecto.py` y el módulo existente `snowflake_io.py` en el mismo directorio. La primera consulta usa las credenciales de Snowflake configuradas mediante variables de entorno. En las ejecuciones posteriores se reutiliza la muestra CSV local, salvo que cambies expresamente `REFRESCAR_DESDE_SNOWFLAKE`.

Usa el kernel remoto **PySpark 4 + Snowflake** del contenedor `spark`. La imagen instala MLflow 2.17.2 y scikit-learn desde `src/res/config/spark/requirements.txt`; reconstruye la imagen Spark después de cambiar ese archivo. El servidor MLflow de Compose recibe las corridas mediante `MLFLOW_TRACKING_URI` y conserva la base y los artefactos en `src/main/mlflow`. Su interfaz está disponible en `http://127.0.0.1:5000` (o el puerto `MLFLOW_PORT` configurado).

In [1]:
#? Importar las librerías necesarias para el experimento.
import os
import sys
import time
import json
import hashlib
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, auc,
    fbeta_score, precision_score, recall_score,
)

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("MLflow:", mlflow.__version__)

Python: 3.10.12
scikit-learn: 1.7.2
MLflow: 2.17.2


### 0.1 Definir funciones y constantes del proyecto
Dado que se esta trabajando bajo un contendor de Apache Spark para utilizar el backend de Snowflake, utilizar librerias
de un modulo que no se encuentre en PiPy o en la libreria estandar es imposible, por tanto, las funciones que se
definian externamente las copio aqui para que puedan usarse en el notebook.

In [2]:
"""Lectura reproducible y división temporal para SSD MC1.

No vuelve a calcular ventanas SMART, no imputa datos y no ajusta estimadores.
El notebook de experimentos importa estas funciones.
"""
from __future__ import annotations

import hashlib
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

SEMILLA = 42
OBJETIVO = "TARGET_30D"
SMART = ("R_1", "N_1", "R_5", "N_5", "R_9", "N_9")
VENTANAS = (7, 14, 30)
TIPOS = ("MEAN", "COUNT", "MEAN_SHIFT")
NUMERICAS = [f"{s}_{t}_{d}D" for s in SMART for d in VENTANAS for t in TIPOS]
PARTICIONES = ("ENTRENAMIENTO", "VALIDACION", "PRUEBA")
MAX_FILAS = 50_000


def sha256_archivo(ruta: str | Path) -> str:
    """Huella SHA-256 de los bytes de un archivo local."""
    h = hashlib.sha256()
    with open(ruta, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()


def nombre_obt() -> str:
    """Nombre de la tabla compartida, sin credenciales dentro del código."""
    base = os.environ["SNOWFLAKE_DATABASE"].strip()
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_$]*", base):
        raise ValueError("SNOWFLAKE_DATABASE no es un identificador válido")
    return f'"{base}"."S_CDATOS_PSET2_OBT"."OBT_MC1_RN"'


def construir_consulta(obt: str) -> str:
    """Reproduce el muestreo estratificado temporal establecido en el proyecto.

    Se seleccionan identificadores SSD-día antes de recuperar las 54 features.
    Las cuotas son máximas. El orden es determinista para una OBT fija.
    """
    columnas = ",\n        ".join(f"o.{c}" for c in NUMERICAS)
    return f"""
WITH candidatos AS (
    -- Fechas utilizables con etiquetas conocidas y embargos de julio y octubre.
    SELECT SSD_KEY, OBSERVATION_DATE, TARGET_30D,
           CASE
             WHEN OBSERVATION_DATE < '2019-07-01'::DATE THEN 'ENTRENAMIENTO'
             WHEN OBSERVATION_DATE BETWEEN '2019-08-01'::DATE AND '2019-09-30'::DATE THEN 'VALIDACION'
             WHEN OBSERVATION_DATE BETWEEN '2019-11-01'::DATE AND '2019-11-30'::DATE THEN 'PRUEBA'
           END AS PARTICION
    FROM {obt}
    WHERE TARGET_30D IN (0, 1)
      AND (OBSERVATION_DATE < '2019-07-01'::DATE
           OR OBSERVATION_DATE BETWEEN '2019-08-01'::DATE AND '2019-09-30'::DATE
           OR OBSERVATION_DATE BETWEEN '2019-11-01'::DATE AND '2019-11-30'::DATE)
),
reducidos AS (
    -- Conservar positivos y toda validación/prueba.
    -- Reducir de forma determinista los negativos de entrenamiento a ~5 %.
    SELECT * FROM candidatos
    WHERE PARTICION IS NOT NULL
      AND (TARGET_30D = 1 OR PARTICION IN ('VALIDACION', 'PRUEBA')
           OR MOD(MOD(HASH(SSD_KEY, OBSERVATION_DATE), 100) + 100, 100) < 5)
),
numerados AS (
    -- Dos rankings: por clase (entrenamiento) y por partición (val/prueba).
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY PARTICION, TARGET_30D
             ORDER BY HASH(SSD_KEY, OBSERVATION_DATE), SSD_KEY, OBSERVATION_DATE
           ) AS FILA_CLASE,
           ROW_NUMBER() OVER (
             PARTITION BY PARTICION
             ORDER BY HASH(SSD_KEY, OBSERVATION_DATE), SSD_KEY, OBSERVATION_DATE
           ) AS FILA_PARTICION
    FROM reducidos
),
seleccion AS (
    -- Hasta 5 000 positivos + 25 000 negativos en entrenamiento.
    -- Hasta 10 000 de validación y 10 000 de prueba sin alterar sus clases.
    SELECT SSD_KEY, OBSERVATION_DATE, TARGET_30D, PARTICION
    FROM numerados
    WHERE (PARTICION = 'ENTRENAMIENTO'
           AND ((TARGET_30D = 1 AND FILA_CLASE <= 5000)
                OR (TARGET_30D = 0 AND FILA_CLASE <= 25000)))
       OR (PARTICION IN ('VALIDACION', 'PRUEBA') AND FILA_PARTICION <= 10000)
)
SELECT o.SSD_KEY, o.OBSERVATION_DATE,
       {columnas},
       s.TARGET_30D, s.PARTICION
FROM seleccion s
JOIN {obt} o
  ON o.SSD_KEY = s.SSD_KEY AND o.OBSERVATION_DATE = s.OBSERVATION_DATE
""".strip()


def consultar_snowflake(sql: str, max_filas: int = MAX_FILAS) -> pd.DataFrame:
    """Replica la conexión Spark/Snowflake del notebook anterior."""
    from snowflake_io import create_spark_session, read_snowflake_query

    texto = sql.strip().rstrip(";")
    if not texto.upper().startswith(("SELECT", "WITH")) or ";" in texto:
        raise ValueError("La consulta debe ser SELECT o WITH, sin instrucciones adicionales")
    spark = create_spark_session("mc1-experimentos-mlflow")
    # El LIMIT exterior es un resguardo del equipo local, no el muestreo.
    resultado = read_snowflake_query(
        spark,
        f"SELECT * FROM ({texto}) q LIMIT {max_filas + 1}",
        schema="S_CDATOS_PSET2_OBT",
    ).toPandas()
    if len(resultado) > max_filas:
        raise ValueError(f"La consulta superó el máximo de {max_filas:,} observaciones")
    return resultado


def normalizar_tipos(datos: pd.DataFrame) -> pd.DataFrame:
    """Solo convierte tipos, sin imputación ni ingeniería adicional."""
    datos = datos.copy()
    requeridas = ["SSD_KEY", "OBSERVATION_DATE", *NUMERICAS, OBJETIVO, "PARTICION"]
    faltantes = sorted(set(requeridas) - set(datos.columns))
    if faltantes:
        raise ValueError(f"Faltan columnas de la OBT: {faltantes}")
    datos["SSD_KEY"] = datos["SSD_KEY"].astype("string")
    datos["OBSERVATION_DATE"] = pd.to_datetime(datos["OBSERVATION_DATE"], errors="raise")
    datos[NUMERICAS] = datos[NUMERICAS].apply(pd.to_numeric, errors="coerce")
    datos[NUMERICAS] = datos[NUMERICAS].replace([np.inf, -np.inf], np.nan)
    datos[OBJETIVO] = pd.to_numeric(datos[OBJETIVO], errors="raise").astype(int)
    if not set(datos[OBJETIVO].unique()).issubset({0, 1}):
        raise ValueError("Se encontró TARGET_30D fuera de 0/1")
    if not set(datos["PARTICION"].unique()).issubset(set(PARTICIONES)):
        raise ValueError("Partición desconocida")
    if datos.duplicated(["SSD_KEY", "OBSERVATION_DATE"]).any():
        raise ValueError("La OBT contiene pares SSD_KEY, OBSERVATION_DATE duplicados")
    return datos


def obtener_muestra(ruta_csv: str | Path, refrescar: bool = False) -> tuple[pd.DataFrame, str]:
    """Congela la muestra localmente y evita cambiarla accidentalmente entre corridas."""
    ruta = Path(ruta_csv)
    if ruta.exists() and not refrescar:
        datos = pd.read_csv(ruta, dtype={"SSD_KEY": "string"}, low_memory=False)
        fuente = "archivo_congelado"
    else:
        datos = consultar_snowflake(construir_consulta(nombre_obt()))
        datos = normalizar_tipos(datos)
        datos = datos.sort_values(
            ["PARTICION", "SSD_KEY", "OBSERVATION_DATE"], kind="stable"
        ).reset_index(drop=True)
        ruta.parent.mkdir(parents=True, exist_ok=True)
        datos.to_csv(ruta, index=False, date_format="%Y-%m-%d", float_format="%.17g")
        fuente = "snowflake"
    return normalizar_tipos(datos), fuente


def dividir_por_fecha(datos: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Respeta la partición de SQL y verifica sus límites cronológicos."""
    particiones = {
        p: datos.loc[datos["PARTICION"] == p].copy()
        for p in PARTICIONES
    }
    if any(df.empty for df in particiones.values()):
        raise ValueError("Una partición quedó vacía")
    limites = {
        "ENTRENAMIENTO": (None, "2019-06-30"),
        "VALIDACION": ("2019-08-01", "2019-09-30"),
        "PRUEBA": ("2019-11-01", "2019-11-30"),
    }
    for nombre, conjunto in particiones.items():
        desde, hasta = limites[nombre]
        if desde and (conjunto["OBSERVATION_DATE"] < pd.Timestamp(desde)).any():
            raise ValueError(f"Fecha inválida en {nombre}")
        if (conjunto["OBSERVATION_DATE"] > pd.Timestamp(hasta)).any():
            raise ValueError(f"Fecha inválida en {nombre}")
        if set(conjunto[OBJETIVO].unique()) != {0, 1}:
            raise ValueError(f"La partición {nombre} no contiene las dos clases")
    return particiones


def seleccionar_caracteristicas(
    entrenamiento: pd.DataFrame, max_fraccion_nulos: float = 0.05
) -> tuple[list[str], pd.DataFrame]:
    """Filtra columnas usando SOLO entrenamiento: nulos y varianza nula."""
    analisis = pd.DataFrame({
        "caracteristica": NUMERICAS,
        "fraccion_nulos_entrenamiento": entrenamiento[NUMERICAS].isna().mean().values,
        "valores_distintos_entrenamiento": entrenamiento[NUMERICAS].nunique(dropna=True).values,
    })
    analisis["utilizar"] = (
        (analisis["fraccion_nulos_entrenamiento"] <= max_fraccion_nulos)
        & (analisis["valores_distintos_entrenamiento"] > 1)
    )
    elegidas = analisis.loc[analisis["utilizar"], "caracteristica"].tolist()
    if len(elegidas) < 2:
        raise ValueError("Quedaron menos de dos características. Revisar nulos")
    return elegidas, analisis


def preparar_conjuntos_sin_imputacion(
    particiones: dict[str, pd.DataFrame], columnas: list[str]
) -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    """Elimina solo SSD-día incompletos en las columnas elegidas.

    Los cuatro modelos reciben las mismas filas por partición.
    No se inventan lecturas SMART para rellenar ventanas incompletas.
    """
    preparados = {}
    registros = []
    for nombre, datos in particiones.items():
        limpios = datos.dropna(subset=columnas).copy()
        if set(limpios[OBJETIVO].unique()) != {0, 1}:
            raise ValueError(f"La limpieza dejó una clase ausente en {nombre}")
        preparados[nombre] = limpios
        registros.append({
            "particion": nombre,
            "filas_antes": len(datos),
            "filas_despues": len(limpios),
            "filas_eliminadas": len(datos) - len(limpios),
            "positivos": int((limpios[OBJETIVO] == 1).sum()),
            "negativos": int((limpios[OBJETIVO] == 0).sum()),
            "ssd_positivos_unicos": int(limpios.loc[limpios[OBJETIVO] == 1, "SSD_KEY"].nunique()),
        })
    return preparados, pd.DataFrame(registros)


def separar_xy(
    particiones: dict[str, pd.DataFrame], columnas: list[str]
) -> dict[str, tuple[pd.DataFrame, pd.Series]]:
    """Separa X e y y excluye fecha, SSD_KEY y etiquetas futuras."""
    return {
        nombre: (datos[columnas].copy(), datos[OBJETIVO].copy())
        for nombre, datos in particiones.items()
    }


### 0.2. Configurar la ejecución

Las rutas relativas hacen que el notebook guarde los archivos junto a `pipeline_proyecto.py`. `REFRESCAR_DESDE_SNOWFLAKE=False` reutiliza los mismos registros que la ejecución anterior, en lugar de seleccionar una muestra nueva.

In [3]:
#? Parámetros únicos de la ejecución. No incluir credenciales en el código.
DIRECTORIO = Path.cwd()
ARCHIVO_DATOS = DIRECTORIO / "mc1_muestra_experimentos.csv"
ARCHIVO_CORRIDAS = DIRECTORIO / "corridas_proyecto.csv"
ARCHIVO_MANIFIESTO = DIRECTORIO / "mc1_manifiesto_experimento.json"
REFRESCAR_DESDE_SNOWFLAKE = False
MAX_FRACCION_NULOS = 0.05
NOMBRE_EXPERIMENTO = "Antes de que falle - SSD MC1 - TARGET_30D"
NOMBRE_REGISTRADO = "antes_de_que_falle_mc1_clasificador"

print("Directorio:", DIRECTORIO)
print("Actualizar muestra desde Snowflake:", REFRESCAR_DESDE_SNOWFLAKE)

Directorio: /opt/spark/notebooks
Actualizar muestra desde Snowflake: False


## 1. Leer la OBT o recuperar una muestra congelada

La consulta selecciona **hasta 5.000 positivos y 25.000 negativos para entrenamiento**. Valida y prueba con un máximo de 10.000 observaciones cada uno, sin cuotas por clase. Se utilizan las particiones ya acordadas: entrenamiento hasta junio de 2019, validación en agosto y septiembre, y prueba en noviembre, con julio y octubre excluidos.

La muestra local se guarda antes de entrenar. La tabla compartida sigue siendo de solo lectura. La consulta puede ser costosa porque Snowflake debe ordenar candidatos para cumplir las cuotas, aunque solo descargue 50.000 registros.

In [5]:
#? Leer el archivo existente o ejecutar la consulta en Snowflake una sola vez.
datos, fuente_datos = obtener_muestra(
    ARCHIVO_DATOS,
    refrescar=REFRESCAR_DESDE_SNOWFLAKE,
)
print("Origen de esta ejecución:", fuente_datos)
print("Observaciones:", len(datos))
print("Columnas:", len(datos.columns))

26/11/07 02:28:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Origen de esta ejecución: snowflake
Observaciones: 50000
Columnas: 58


### 1.1. Revisar los registros obtenidos

In [6]:
#? La tabla queda como DataFrame para Data Wrangler.
datos.head(30)

,SSD_KEY,OBSERVATION_DATE,R_1_MEAN_7D,R_1_COUNT_7D,R_1_MEAN_SHIFT_7D,R_1_MEAN_14D,R_1_COUNT_14D,R_1_MEAN_SHIFT_14D,R_1_MEAN_30D,R_1_COUNT_30D,...,N_9_COUNT_7D,N_9_MEAN_SHIFT_7D,N_9_MEAN_14D,N_9_COUNT_14D,N_9_MEAN_SHIFT_14D,N_9_MEAN_30D,N_9_COUNT_30D,N_9_MEAN_SHIFT_30D,TARGET_30D,PARTICION
0,000035c97d82c585d0ae0a09dd82b7ed26d5f87f1c2e07...,2018-08-10,0.000000,7.0,0.000000,0.000000,14.0,0.000000,0.000000,29.0,...,7.0,0.0,100.0,14.0,0.0,100.0,29.0,0.0,0,ENTRENAMIENTO
1,0004488a91d62006df7e4865aead4188a8a1d0a0d106f1...,2018-08-10,0.000000,7.0,0.000000,0.000000,14.0,0.000000,0.000000,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
2,0005fa30cc4140cd15ae6d3ce893d70143318e021c162c...,2018-10-27,0.000000,7.0,0.000000,0.000000,14.0,0.000000,0.000000,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
3,000ed4a53f671f0660d70daeefa70b5ab688c5038d2fca...,2018-04-03,0.000000,7.0,0.000000,0.000000,14.0,0.000000,0.000000,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
4,0010224ca508a4c5f4fddfa0655c611c2b4f1859349eaa...,2019-02-09,0.000000,7.0,0.000000,0.000000,14.0,0.000000,0.000000,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
5,00102dc1ef41ba97c6ab8d304926e11d209f5a8c8040f3...,2018-06-23,0.000000,7.0,0.000000,0.000000,13.0,NaN,0.000000,13.0,...,7.0,0.0,100.0,13.0,NaN,100.0,13.0,NaN,0,ENTRENAMIENTO
6,0014159c1a1e03eff88836223491150ae32a552eb8f2f3...,2018-12-18,0.000000,7.0,0.000000,0.000000,14.0,0.000000,0.000000,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
7,001e40daecb696f53acc03d0282d9f1b249702d3c28acc...,2018-07-20,0.000000,7.0,0.000000,0.000000,14.0,0.000000,0.000000,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
8,001e40daecb696f53acc03d0282d9f1b249702d3c28acc...,2018-09-15,0.000000,7.0,0.000000,0.000000,14.0,0.000000,0.000000,30.0,...,7.0,0.0,100.0,14.0,0.0,100.0,30.0,0.0,0,ENTRENAMIENTO
9,001e6ca888ecac8d99451307355b448e3ad01813327898...,2018-11-22,0.000000,7.0,0.000000,0.000000,10.0,0.000000,0.000000,26.0,...,7.0,0.0,100.0,10.0,0.0,100.0,26.0,0.0,0,ENTRENAMIENTO


### 1.2. Comprobar el desbalance por partición

In [7]:
#? Contar las dos clases sin alterar la distribución.
distribucion_original = pd.crosstab(
    datos["PARTICION"], datos[OBJETIVO]
)
distribucion_original["TOTAL"] = distribucion_original.sum(axis=1)
distribucion_original

TARGET_30D,0,1,TOTAL
PARTICION,,,
ENTRENAMIENTO,25000,5000,30000
PRUEBA,9959,41,10000
VALIDACION,9936,64,10000


## 2. Dividir cronológicamente los conjuntos

La función importada `dividir_por_fecha()` valida los límites de cada partición. No se usa `train_test_split`, porque el experimento simula predicciones sobre fechas futuras. Un SSD puede aparecer en más de una partición, pero solo se utilizan sus observaciones disponibles en cada fecha.

In [8]:
#? Dividir y validar las fechas, sin mezclar observaciones.
particiones_originales = dividir_por_fecha(datos)
for nombre, conjunto in particiones_originales.items():
    print(nombre, len(conjunto), "desde", conjunto["OBSERVATION_DATE"].min().date(),
          "hasta", conjunto["OBSERVATION_DATE"].max().date())

ENTRENAMIENTO 30000 desde 2018-01-03 hasta 2019-06-30
VALIDACION 10000 desde 2019-08-01 hasta 2019-09-30
PRUEBA 10000 desde 2019-11-01 hasta 2019-11-30


### 2.1. Seleccionar características usando solo entrenamiento

Se parte de las **54 características** disponibles. Se descartan únicamente las columnas con más de un 5 % de valores ausentes en entrenamiento o que sean constantes. No se consulta validación ni prueba para elegir columnas. Este filtro no sustituye el análisis de atributos SMART ni crea lecturas artificiales.

In [9]:
#? Aprender las columnas utilizables a partir de entrenamiento solamente.
columnas_modelo, auditoria_columnas = seleccionar_caracteristicas(
    particiones_originales["ENTRENAMIENTO"],
    max_fraccion_nulos=MAX_FRACCION_NULOS,
)
print("Columnas disponibles:", len(NUMERICAS))
print("Columnas seleccionadas:", len(columnas_modelo))
print("Columnas descartadas:", len(NUMERICAS) - len(columnas_modelo))

Columnas disponibles: 54
Columnas seleccionadas: 41
Columnas descartadas: 13


### 2.2. Examinar las columnas conservadas y descartadas

In [10]:
#? Muestra el porcentaje de nulos y la decisión tomada.
auditoria_columnas.sort_values(
    ["utilizar", "fraccion_nulos_entrenamiento"],
    ascending=[True, False],
)

,caracteristica,fraccion_nulos_entrenamiento,valores_distintos_entrenamiento,utilizar
8,R_1_MEAN_SHIFT_30D,0.062367,1035,False
17,N_1_MEAN_SHIFT_30D,0.062367,1,False
26,R_5_MEAN_SHIFT_30D,0.062367,1137,False
35,N_5_MEAN_SHIFT_30D,0.062367,77,False
44,R_9_MEAN_SHIFT_30D,0.062367,7513,False
53,N_9_MEAN_SHIFT_30D,0.062367,1,False
14,N_1_MEAN_SHIFT_14D,0.030800,1,False
50,N_9_MEAN_SHIFT_14D,0.030800,1,False
11,N_1_MEAN_SHIFT_7D,0.016600,1,False
47,N_9_MEAN_SHIFT_7D,0.016600,1,False


### 2.3. Retirar los SSD-día incompletos, sin imputarlos

Los modelos reciben las **mismas observaciones limpias**, incluso la red neuronal. Si una fila tiene un valor ausente en alguna de las características seleccionadas, se excluye. Este criterio puede reducir el número de ejemplos positivos. El notebook lo cuantifica para evitar ocultar esa pérdida.

La OBT ya contiene las ventanas temporales. Esta etapa no calcula medias, diferencias ni valores SMART nuevos.

In [11]:
#? Eliminar filas incompletas de manera idéntica en los tres conjuntos.
particiones, resumen_filtrado = preparar_conjuntos_sin_imputacion(
    particiones_originales, columnas_modelo
)
resumen_filtrado

,particion,filas_antes,filas_despues,filas_eliminadas,positivos,negativos,ssd_positivos_unicos
0,ENTRENAMIENTO,30000,29032,968,4916,24116,3190
1,VALIDACION,10000,9798,202,64,9734,64
2,PRUEBA,10000,9782,218,40,9742,39


### 2.4. Separar entradas y etiquetas

`SSD_KEY`, `OBSERVATION_DATE`, `PARTICION` y los atributos sobre el futuro permanecen fuera de `X`. La etiqueta `TARGET_30D` solo se utiliza como `y` para entrenar y evaluar.

In [12]:
#? Crear los tres pares X e y con exactamente las mismas columnas.
pares_xy = separar_xy(particiones, columnas_modelo)
X_ent, y_ent = pares_xy["ENTRENAMIENTO"]
X_val, y_val = pares_xy["VALIDACION"]
X_prueba, y_prueba = pares_xy["PRUEBA"]

print("X entrenamiento:", X_ent.shape)
print("X validación:", X_val.shape)
print("X prueba:", X_prueba.shape)
print("Nulos en X:", X_ent.isna().sum().sum(), X_val.isna().sum().sum(), X_prueba.isna().sum().sum())

X entrenamiento: (29032, 41)
X validación: (9798, 41)
X prueba: (9782, 41)
Nulos en X: 0 0 0


## 3. Registrar la procedencia de los datos

La huella SHA-256 identifica el contenido exacto de `mc1_muestra_experimentos.csv`. También registramos la consulta, los límites temporales, la selección de características, las filas eliminadas y las versiones de las bibliotecas. Una huella permite detectar cambios, pero **no reemplaza el archivo**. Para que otra persona reproduzca el experimento debe recibir la misma muestra o poder obtenerla desde la misma versión de la OBT.

In [14]:
#? Calcular las huellas de los archivos y del SQL utilizado.
huella_datos = sha256_archivo(ARCHIVO_DATOS)
try:
    huella_sql = hashlib.sha256(
        construir_consulta(nombre_obt()).encode("utf-8")
    ).hexdigest()
    nombre_tabla = nombre_obt()
except KeyError:
    #? Es posible reproducir desde la muestra local sin credenciales Snowflake.
    huella_sql = "sin_entorno_snowflake"
    nombre_tabla = "OBT_MC1_RN (nombre completo no disponible)"

print("SHA-256 datos:", huella_datos)
print("SHA-256 SQL:", huella_sql)

SHA-256 datos: 9fe1fcb4da4c4d099e4e989fdfbc7b7cb537f21d1f4e965aaf94bb64c42554ee
SHA-256 SQL: d700880b1d86bb77a100955a2d335347ea16aba9962e17ab0202678bdd198de1


### 3.1. Crear un manifiesto de la ejecución

In [15]:
#? Guardar la configuración sin incluir claves, contraseñas ni tokens.
manifiesto = {
    "proyecto": NOMBRE_EXPERIMENTO,
    "tabla_origen": nombre_tabla,
    "muestra_archivo": ARCHIVO_DATOS.name,
    "muestra_sha256": huella_datos,
    "sql_sha256": huella_sql,
    "semilla": SEMILLA,
    "muestreo": "ent: 5000 positivos + 25000 negativos; val: 10000; prueba: 10000",
    "fechas": {
        "entrenamiento_fin": "2019-06-30",
        "embargo_1": "2019-07",
        "validacion_inicio": "2019-08-01",
        "validacion_fin": "2019-09-30",
        "embargo_2": "2019-10",
        "prueba_inicio": "2019-11-01",
        "prueba_fin": "2019-11-30",
    },
    "caracteristicas_candidatas": NUMERICAS,
    "caracteristicas_seleccionadas": columnas_modelo,
    "max_fraccion_nulos_entrenamiento": MAX_FRACCION_NULOS,
    "filas_finales": resumen_filtrado.to_dict(orient="records"),
    "versiones": {
        "python": sys.version.split()[0],
        "scikit_learn": sklearn.__version__,
        "pandas": pd.__version__,
        "mlflow": mlflow.__version__,
        "numpy": np.__version__,
        "plataforma": platform.platform(),
    },
}
ARCHIVO_MANIFIESTO.write_text(
    json.dumps(manifiesto, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Manifiesto guardado:", ARCHIVO_MANIFIESTO.name)

Manifiesto guardado: mc1_manifiesto_experimento.json


## 4. Preparar MLflow

El servidor MLflow utiliza **SQLite** en `src/main/mlflow` para conservar las corridas y el registro del modelo final. También sirve los artefactos por HTTP. El experimento tiene un nombre fijo. Cada corrida registra los hiperparámetros, las métricas de entrenamiento, validación y prueba, la duración de `fit()`, las huellas y el pipeline entrenado.

La métrica principal se registra como **PR-AUC (Average Precision)**, una medida sin interpolación de la curva precisión–recall adecuada para el desbalance extremo. También se guarda el área trapezoidal con otro nombre, porque puede diferir mucho de AP cuando todos los puntajes son iguales, como ocurre con la línea base. Se registran F0.5, precisión y recall con un umbral fijo de 0.5. El punto operativo definitivo y la evaluación por eventos SSD quedan fuera de este ejercicio de registro.

In [16]:
#? Conectar con el servidor de Compose y crear o reutilizar el experimento.
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment(NOMBRE_EXPERIMENTO)
experimento = mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO)
print("Experimento:", experimento.name)
print("Experiment ID:", experimento.experiment_id)
print("Tracking URI:", mlflow.get_tracking_uri())

2026/11/07 02:31:58 INFO mlflow.tracking.fluent: Experiment with name 'Antes de que falle - SSD MC1 - TARGET_30D' does not exist. Creating a new experiment.


Experimento: Antes de que falle - SSD MC1 - TARGET_30D
Experiment ID: 1
Tracking URI: http://mlflow:5000


### 4.1. Definir la función de métricas

Para mantener una comparación justa, **todos los modelos se evalúan con las mismas filas** de cada partición. Se calculan las métricas a partir de probabilidades y predicciones de clasificación, sin utilizar los identificadores SSD como predictores.

In [17]:
#? Una función única para calcular métricas de clasificación binaria.
def calcular_metricas(modelo, X, y, prefijo):
    probabilidades = modelo.predict_proba(X)[:, 1]
    predicciones = (probabilidades >= 0.5).astype(int)
    precision_curva, recall_curva, _ = precision_recall_curve(y, probabilidades)
    return {
        f"{prefijo}_pr_auc": float(average_precision_score(y, probabilidades)),
        f"{prefijo}_pr_auc_trapezoidal": float(auc(recall_curva, precision_curva)),
        f"{prefijo}_average_precision": float(average_precision_score(y, probabilidades)),
        f"{prefijo}_f05": float(fbeta_score(y, predicciones, beta=0.5, zero_division=0)),
        f"{prefijo}_precision": float(precision_score(y, predicciones, zero_division=0)),
        f"{prefijo}_recall": float(recall_score(y, predicciones, zero_division=0)),
    }

### 4.2. Definir el registro de una corrida

La función siguiente crea una corrida MLflow, entrena un pipeline y guarda el modelo junto con sus parámetros. Los modelos de árboles reciben directamente las características calculadas en la OBT. **Solo la red neuronal utiliza `StandardScaler`**, ajustado dentro del pipeline y exclusivamente con entrenamiento. Esta escala es necesaria por las magnitudes distintas de las mediciones, y no implica una nueva imputación ni la reconstrucción de las ventanas SMART.

In [20]:
#? Entrenar y registrar un modelo con exactamente la misma muestra y las mismas métricas.
def registrar_corrida(nombre, pipeline):
    with mlflow.start_run(run_name=nombre) as corrida:
        mlflow.set_tag("candidato", "no")
        mlflow.set_tag("proyecto", "SSD_MC1_TARGET_30D")
        mlflow.set_tag("división", "cronológica_embargo_julio_octubre_2019")
        mlflow.set_tag("muestra", "estratificada_entrenamiento")
        mlflow.set_tag("modelo_familia", nombre)

        parametros = pipeline.named_steps["modelo"].get_params(deep=False)
        mlflow.log_params({f"modelo__{k}": str(v) for k, v in parametros.items()})
        mlflow.log_params({
            "semilla": SEMILLA,
            "fecha_entrenamiento_fin": "2019-06-30",
            "fecha_validacion_inicio": "2019-08-01",
            "fecha_validacion_fin": "2019-09-30",
            "fecha_prueba_inicio": "2019-11-01",
            "fecha_prueba_fin": "2019-11-30",
            "embargos": "julio_y_octubre_2019",
            "datos_sha256": huella_datos,
            "sql_sha256": huella_sql,
            "cantidad_features": len(columnas_modelo),
            "filas_entrenamiento": len(X_ent),
            "filas_validacion": len(X_val),
            "filas_prueba": len(X_prueba),
            "version_sklearn": sklearn.__version__,
            "version_mlflow": mlflow.__version__,
        })

        inicio = time.perf_counter()
        pipeline.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio
        metricas = {"segundos_entrenamiento": float(segundos)}
        metricas.update(calcular_metricas(pipeline, X_ent, y_ent, "entrenamiento"))
        metricas.update(calcular_metricas(pipeline, X_val, y_val, "validacion"))
        metricas.update(calcular_metricas(pipeline, X_prueba, y_prueba, "prueba"))
        mlflow.log_metrics(metricas)

        mlflow.log_artifact(str(ARCHIVO_MANIFIESTO), artifact_path="procedencia")
        firma = infer_signature(X_ent.head(5), pipeline.predict(X_ent.head(5)))
        info_modelo = mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path="pipeline_entrenado",
            signature=firma,
            input_example=X_ent.head(5),
        )
        mlflow.set_tag("modelo_uri", info_modelo.model_uri)
        print(f"{nombre}: run_id={corrida.info.run_id}")
        print(f"  PR-AUC validación={metricas['validacion_pr_auc']:.5f}, "
              f"PR-AUC prueba={metricas['prueba_pr_auc']:.5f}, "
              f"segundos={segundos:.2f}")
        return {
            "modelo": nombre, "run_id": corrida.info.run_id,
            "modelo_uri": info_modelo.model_uri, **metricas,
        }

## 5. Entrenar los cuatro clasificadores

El problema de MC1 es **clasificación**, no regresión. Se incluye un clasificador que predice siempre la clase más frecuente, un bosque aleatorio como alternativa al modelo lineal, un boosting de árboles y una red neuronal. Cada celda entrena un solo modelo y crea **una única corrida MLflow**.

En caso de repetir esta sección sin borrar las corridas anteriores, MLflow conservará el historial. Antes de la entrega, elimina las corridas de prueba desde su interfaz, reinicia el kernel y ejecuta todo una sola vez.

### 5.1. Línea base — `DummyClassifier`

Predice la clase mayoritaria. En una población con pocas fallas puede conseguir mucha exactitud, pero su capacidad para priorizar fallas suele ser limitada. No se utiliza la exactitud para elegir el modelo.

In [21]:
#? Modelo 1: línea base, sin transformaciones adicionales.
corridas_ejecucion = []
modelo_base = Pipeline([
    ("modelo", DummyClassifier(strategy="most_frequent"))
])
resultado_base = registrar_corrida("linea_base_dummy", modelo_base)
corridas_ejecucion.append(resultado_base)

2026/11/07 02:33:31 INFO mlflow.tracking._tracking_service.client: 🏃 View run linea_base_dummy at: http://mlflow:5000/#/experiments/1/runs/fc3d126609004fe4921f8f89239f93ec.
2026/11/07 02:33:31 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


linea_base_dummy: run_id=fc3d126609004fe4921f8f89239f93ec
  PR-AUC validación=0.00653, PR-AUC prueba=0.00409, segundos=0.00


### 5.2. Bosque aleatorio — `RandomForestClassifier`

Este modelo sustituye al modelo lineal solicitado en la actividad. Se limita su profundidad y se emplean pesos por clase para que las fallas minoritarias no queden ignoradas. No se aplica un escalador ni se rellenan mediciones SMART.

In [22]:
#? Modelo 2: bosque aleatorio con complejidad acotada.
modelo_bosque = Pipeline([
    ("modelo", RandomForestClassifier(
        n_estimators=120,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=SEMILLA,
        n_jobs=1,
    ))
])
resultado_bosque = registrar_corrida("bosque_aleatorio", modelo_bosque)
corridas_ejecucion.append(resultado_bosque)

2026/11/07 02:33:43 INFO mlflow.tracking._tracking_service.client: 🏃 View run bosque_aleatorio at: http://mlflow:5000/#/experiments/1/runs/e0bacb382fd247dc9ba1a9f854755e56.
2026/11/07 02:33:43 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


bosque_aleatorio: run_id=e0bacb382fd247dc9ba1a9f854755e56
  PR-AUC validación=0.01619, PR-AUC prueba=0.03707, segundos=1.87


### 5.3. Ensamblaje por boosting — `HistGradientBoostingClassifier`

El boosting construye árboles secuencialmente para reducir errores. Se controla la complejidad con hojas limitadas, regularización y un número moderado de iteraciones.

In [23]:
#? Modelo 3: boosting de árboles de clasificación.
modelo_boosting = Pipeline([
    ("modelo", HistGradientBoostingClassifier(
        max_iter=120,
        learning_rate=0.05,
        max_leaf_nodes=15,
        min_samples_leaf=20,
        l2_regularization=1.0,
        random_state=SEMILLA,
    ))
])
resultado_boosting = registrar_corrida("gradient_boosting", modelo_boosting)
corridas_ejecucion.append(resultado_boosting)

2026/11/07 02:33:49 INFO mlflow.tracking._tracking_service.client: 🏃 View run gradient_boosting at: http://mlflow:5000/#/experiments/1/runs/113da9a0fc62477ead058c67b0cdae5a.
2026/11/07 02:33:49 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


gradient_boosting: run_id=113da9a0fc62477ead058c67b0cdae5a
  PR-AUC validación=0.02334, PR-AUC prueba=0.02554, segundos=0.37


### 5.4. Red neuronal — `MLPClassifier`

A diferencia de los árboles, una red neuronal suele necesitar que los valores de entrada tengan escalas comparables. **Aquí sí se ajusta `StandardScaler`, solo con entrenamiento**, dentro del mismo artefacto MLflow. No se realiza imputación, ya que todos los clasificadores utilizan las mismas filas completas seleccionadas en el paso 2.

In [24]:
#? Modelo 4: red neuronal con escalado específico y reproducible.
modelo_red = Pipeline([
    ("escalar", StandardScaler()),
    ("modelo", MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        alpha=0.001,
        batch_size=256,
        learning_rate_init=0.001,
        max_iter=120,
        random_state=SEMILLA,
        early_stopping=False,
    )),
])
resultado_red = registrar_corrida("red_neuronal_mlp", modelo_red)
corridas_ejecucion.append(resultado_red)

/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (120) reached and the optimization hasn't converged yet.
  warnings.warn(
2026/11/07 02:33:57 INFO mlflow.tracking._tracking_service.client: 🏃 View run red_neuronal_mlp at: http://mlflow:5000/#/experiments/1/runs/7ed95ce1c1e5471db84ba174a1ab1ba3.
2026/11/07 02:33:57 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


red_neuronal_mlp: run_id=7ed95ce1c1e5471db84ba174a1ab1ba3
  PR-AUC validación=0.03907, PR-AUC prueba=0.01310, segundos=2.71


## 6. Comparar los resultados de esta ejecución

Para elegir el candidato se utiliza **PR-AUC sobre validación**. La columna de prueba se muestra como evidencia para la actividad, pero **no participa en la selección**. F0.5 se registra como métrica complementaria, sin optimizar un umbral sobre prueba.

In [25]:
#? Conservar las cuatro corridas de esta ejecución como DataFrame.
resultados = pd.DataFrame(corridas_ejecucion)
resultados = resultados.sort_values(
    ["validacion_pr_auc", "validacion_f05"], ascending=False
).reset_index(drop=True)
resultados[[
    "modelo", "run_id",
    "entrenamiento_pr_auc",
    "validacion_pr_auc",
    "prueba_pr_auc",
    "validacion_average_precision",
    "validacion_f05", "prueba_f05", "segundos_entrenamiento",
]]

,modelo,run_id,entrenamiento_pr_auc,validacion_pr_auc,prueba_pr_auc,validacion_average_precision,validacion_f05,prueba_f05,segundos_entrenamiento
0,red_neuronal_mlp,7ed95ce1c1e5471db84ba174a1ab1ba3,0.574911,0.039071,0.013095,0.039071,0.021067,0.010020,2.705023
1,gradient_boosting,113da9a0fc62477ead058c67b0cdae5a,0.600282,0.023344,0.025542,0.023344,0.048343,0.035545,0.372349
2,bosque_aleatorio,e0bacb382fd247dc9ba1a9f854755e56,0.672912,0.016190,0.037071,0.016190,0.009945,0.006300,1.870467
3,linea_base_dummy,fc3d126609004fe4921f8f89239f93ec,0.169330,0.006532,0.004089,0.006532,0.000000,0.000000,0.000688


### 6.1. Elegir el candidato final sin consultar la métrica de prueba

In [26]:
#? La decisión usa únicamente el desempeño de validación.
seleccionado = resultados.iloc[0]
run_id_final = str(seleccionado["run_id"])
modelo_final_nombre = str(seleccionado["modelo"])
modelo_final_uri = str(seleccionado["modelo_uri"])

print("Modelo elegido:", modelo_final_nombre)
print("run_id:", run_id_final)
print("PR-AUC validación:", round(float(seleccionado["validacion_pr_auc"]), 6))
print("F0.5 validación:", round(float(seleccionado["validacion_f05"]), 6))

Modelo elegido: red_neuronal_mlp
run_id: 7ed95ce1c1e5471db84ba174a1ab1ba3
PR-AUC validación: 0.039071
F0.5 validación: 0.021067


### 6.2. Marcar la corrida final y registrar el modelo

La etiqueta `candidato=final` se aplica a la corrida elegida. Además, el pipeline registrado como artefacto se publica en el **Model Registry de MLflow** con un número de versión. La comprobación posterior lo recupera desde `models:/...`, no desde el objeto que permanece en memoria.

In [27]:
#? Marcar la corrida elegida y registrar una versión del modelo.
cliente_mlflow = MlflowClient()
cliente_mlflow.set_tag(run_id_final, "candidato", "final")

version_registrada = mlflow.register_model(
    model_uri=modelo_final_uri,
    name=NOMBRE_REGISTRADO,
)
uri_registro = f"models:/{NOMBRE_REGISTRADO}/{version_registrada.version}"
print("Modelo registrado:", uri_registro)
print("run_id final:", run_id_final)

Successfully registered model 'antes_de_que_falle_mc1_clasificador'.
2026/11/07 02:34:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: antes_de_que_falle_mc1_clasificador, version 1


Modelo registrado: models:/antes_de_que_falle_mc1_clasificador/1
run_id final: 7ed95ce1c1e5471db84ba174a1ab1ba3


Created version '1' of model 'antes_de_que_falle_mc1_clasificador'.


## 7. Cargar el modelo desde MLflow y reproducir la métrica

Se carga la versión publicada en el registro, se vuelve a predecir sobre **la misma prueba congelada** y se recalcula Average Precision. La diferencia con la métrica registrada debe quedar dentro de una tolerancia numérica muy pequeña. No se vuelve a entrenar el modelo.

In [28]:
#? Recuperar exclusivamente el modelo del registro.
modelo_recuperado = mlflow.sklearn.load_model(uri_registro)
print("Tipo recuperado:", type(modelo_recuperado).__name__)
print("Pasos del pipeline recuperado:", list(modelo_recuperado.named_steps))

Tipo recuperado: Pipeline
Pasos del pipeline recuperado: ['escalar', 'modelo']


### 7.1. Verificar la reproducción de la métrica

In [29]:
#? Comprobar PR-AUC de prueba contra el valor de la corrida final.
probabilidades_reproducidas = modelo_recuperado.predict_proba(X_prueba)[:, 1]
pr_auc_recalculada = float(average_precision_score(
    y_prueba, probabilidades_reproducidas
))
pr_auc_registrada = float(
    mlflow.get_run(run_id_final).data.metrics["prueba_pr_auc"]
)
diferencia = abs(pr_auc_recalculada - pr_auc_registrada)

print("PR-AUC registrada:", f"{pr_auc_registrada:.12f}")
print("PR-AUC reproducida:", f"{pr_auc_recalculada:.12f}")
print("Diferencia:", f"{diferencia:.12g}")
assert np.isclose(pr_auc_recalculada, pr_auc_registrada, rtol=1e-9, atol=1e-12), (
    "El modelo recuperado no reproduce la métrica registrada"
)
print("Comprobación: OK")

PR-AUC registrada: 0.013095241279
PR-AUC reproducida: 0.013095241279
Diferencia: 0
Comprobación: OK


## 8. Exportar las corridas desde MLflow

Se consultan las corridas del experimento mediante `mlflow.search_runs()` y se genera `corridas_proyecto.csv` en el directorio del notebook. La tabla contiene los identificadores, las métricas y los parámetros realmente registrados.

Si el experimento ya tenía corridas de prueba, la exportación también las incluirá. Antes de entregar, elimínalas en la interfaz de MLflow, reinicia el kernel y ejecuta el notebook completo.

In [30]:
#? Consultar el experimento real, no construir un CSV manual.
corridas_mlflow = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id],
    output_format="pandas",
)
corridas_mlflow.to_csv("corridas_proyecto.csv", index=False)
print("Archivo generado:", ARCHIVO_CORRIDAS.name)
print("Corridas en el experimento:", len(corridas_mlflow))

Archivo generado: corridas_proyecto.csv
Corridas en el experimento: 5


### 8.1. Inspeccionar la tabla exportada

In [31]:
#? Dejar disponible el DataFrame completo para Data Wrangler.
corridas_mlflow

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.prueba_pr_auc_trapezoidal,metrics.prueba_average_precision,metrics.segundos_entrenamiento,metrics.prueba_recall,...,tags.mlflow.user,tags.proyecto,tags.muestra,tags.mlflow.log-model.history,tags.modelo_familia,tags.modelo_uri,tags.mlflow.source.name,tags.mlflow.runName,tags.división,tags.mlflow.source.type
0,7ed95ce1c1e5471db84ba174a1ab1ba3,1,FINISHED,mlflow-artifacts:/1/7ed95ce1c1e5471db84ba174a1...,2026-11-07 02:33:54.334000+00:00,2026-11-07 02:33:57.917000+00:00,0.009514,0.013095,2.705023,0.100,...,spark,SSD_MC1_TARGET_30D,estratificada_entrenamiento,"[{""run_id"": ""7ed95ce1c1e5471db84ba174a1ab1ba3""...",red_neuronal_mlp,runs:/7ed95ce1c1e5471db84ba174a1ab1ba3/pipelin...,/usr/local/lib/python3.10/dist-packages/ipyker...,red_neuronal_mlp,cronológica_embargo_julio_octubre_2019,LOCAL
1,113da9a0fc62477ead058c67b0cdae5a,1,FINISHED,mlflow-artifacts:/1/113da9a0fc62477ead058c67b0...,2026-11-07 02:33:47.982000+00:00,2026-11-07 02:33:49.292000+00:00,0.020830,0.025542,0.372349,0.150,...,spark,SSD_MC1_TARGET_30D,estratificada_entrenamiento,"[{""run_id"": ""113da9a0fc62477ead058c67b0cdae5a""...",gradient_boosting,runs:/113da9a0fc62477ead058c67b0cdae5a/pipelin...,/usr/local/lib/python3.10/dist-packages/ipyker...,gradient_boosting,cronológica_embargo_julio_octubre_2019,LOCAL
2,e0bacb382fd247dc9ba1a9f854755e56,1,FINISHED,mlflow-artifacts:/1/e0bacb382fd247dc9ba1a9f854...,2026-11-07 02:33:40.793000+00:00,2026-11-07 02:33:43.799000+00:00,0.035708,0.037071,1.870467,0.425,...,spark,SSD_MC1_TARGET_30D,estratificada_entrenamiento,"[{""run_id"": ""e0bacb382fd247dc9ba1a9f854755e56""...",bosque_aleatorio,runs:/e0bacb382fd247dc9ba1a9f854755e56/pipelin...,/usr/local/lib/python3.10/dist-packages/ipyker...,bosque_aleatorio,cronológica_embargo_julio_octubre_2019,LOCAL
3,fc3d126609004fe4921f8f89239f93ec,1,FINISHED,mlflow-artifacts:/1/fc3d126609004fe4921f8f8923...,2026-11-07 02:33:30.150000+00:00,2026-11-07 02:33:31.809000+00:00,0.502045,0.004089,0.000688,0.000,...,spark,SSD_MC1_TARGET_30D,estratificada_entrenamiento,"[{""run_id"": ""fc3d126609004fe4921f8f89239f93ec""...",linea_base_dummy,runs:/fc3d126609004fe4921f8f89239f93ec/pipelin...,/usr/local/lib/python3.10/dist-packages/ipyker...,linea_base_dummy,cronológica_embargo_julio_octubre_2019,LOCAL
4,ef4f1b0b505741c4ae91d52e7fb43681,1,FAILED,mlflow-artifacts:/1/ef4f1b0b505741c4ae91d52e7f...,2026-11-07 02:32:20.118000+00:00,2026-11-07 02:32:20.370000+00:00,0.502045,0.004089,0.000672,0.000,...,spark,SSD_MC1_TARGET_30D,estratificada_entrenamiento,None,linea_base_dummy,None,/usr/local/lib/python3.10/dist-packages/ipyker...,linea_base_dummy,cronológica_embargo_julio_octubre_2019,LOCAL


## 9. Respuestas para la entrega

Las cifras siguientes se generan a partir de la corrida final, sin inventar un `run_id`. Copia el texto después de ejecutar las cuatro corridas y verificar que el modelo registrado reproduce la métrica.

In [32]:
#? Preparar una respuesta basada en los valores reales de MLflow.
respuesta_decision = (
    f"Se eligió {modelo_final_nombre} como modelo final, con run_id {run_id_final}. "
    f"En validación obtuvo PR-AUC={float(seleccionado['validacion_pr_auc']):.6f} "
    f"y F0.5={float(seleccionado['validacion_f05']):.6f}. "
    f"En prueba obtuvo PR-AUC={pr_auc_registrada:.6f}. "
    f"El entrenamiento tomó {float(seleccionado['segundos_entrenamiento']):.3f} segundos. "
    "La elección se realizó a partir de la métrica de validación."
)
print(respuesta_decision)

Se eligió red_neuronal_mlp como modelo final, con run_id 7ed95ce1c1e5471db84ba174a1ab1ba3. En validación obtuvo PR-AUC=0.039071 y F0.5=0.021067. En prueba obtuvo PR-AUC=0.013095. El entrenamiento tomó 2.705 segundos. La elección se realizó a partir de la métrica de validación.


### 9.1. Requisitos para reproducir exactamente las métricas

In [33]:
#? Texto de reproducibilidad y sus limitaciones.
respuesta_reproduccion = (
    "Para reproducir los resultados se necesita la misma muestra de datos, "
    "el módulo pipeline_proyecto.py, la selección de características, los límites "
    "temporales, los hiperparámetros, la semilla y las mismas versiones de Python "
    "y de las bibliotecas. MLflow registra las huellas SHA-256, los parámetros, "
    "las métricas, las versiones y el pipeline entrenado. El archivo CSV congelado "
    "debe conservarse aparte y compartirse de forma autorizada, porque la huella "
    "no contiene las observaciones. También pueden influir las diferencias de "
    "hardware y bibliotecas numéricas, por lo que se comprueba la reproducción "
    "de la métrica al cargar el modelo registrado."
)
print(respuesta_reproduccion)

Para reproducir los resultados se necesita la misma muestra de datos, el módulo pipeline_proyecto.py, la selección de características, los límites temporales, los hiperparámetros, la semilla y las mismas versiones de Python y de las bibliotecas. MLflow registra las huellas SHA-256, los parámetros, las métricas, las versiones y el pipeline entrenado. El archivo CSV congelado debe conservarse aparte y compartirse de forma autorizada, porque la huella no contiene las observaciones. También pueden influir las diferencias de hardware y bibliotecas numéricas, por lo que se comprueba la reproducción de la métrica al cargar el modelo registrado.


## 10. Comprobación final del entregable

Al terminar debes tener `ExperimentosModelos_MC1_Proyecto.ipynb`, `pipeline_proyecto.py`, `mc1_muestra_experimentos.csv`, `mc1_manifiesto_experimento.json` y `corridas_proyecto.csv`. Las corridas, los artefactos y el modelo registrado deben aparecer en la interfaz MLflow; la base y los artefactos persisten en `src/main/mlflow` del host. El CSV de corridas solo existe **después de ejecutar el notebook conectado a tus datos**. No representa resultados ficticios.

In [34]:
#? Comprobar que todos los archivos reproducibles se generaron.
comprobacion = pd.DataFrame([
    {"archivo": p.name, "existe": p.exists(), "bytes": p.stat().st_size if p.exists() else 0}
    for p in [
        DIRECTORIO / "ExperimentosModelos_MC1_Proyecto.ipynb",
        DIRECTORIO / "pipeline_proyecto.py",
        ARCHIVO_DATOS, ARCHIVO_MANIFIESTO, ARCHIVO_CORRIDAS,
    ]
])
comprobacion

,archivo,existe,bytes
0,ExperimentosModelos_MC1_Proyecto.ipynb,False,0
1,pipeline_proyecto.py,False,0
2,mc1_muestra_experimentos.csv,True,13911975
3,mc1_manifiesto_experimento.json,True,3832
4,corridas_proyecto.csv,True,10182
